## Giới thiệu 

Bài học này sẽ bao gồm: 
- Giải thích gọi hàm là gì và các trường hợp sử dụng của nó 
- Cách tạo cuộc gọi hàm sử dụng Azure OpenAI 
- Cách tích hợp cuộc gọi hàm vào một ứng dụng 

## Mục tiêu học tập 

Sau khi hoàn thành bài học này, bạn sẽ biết cách và hiểu về: 

- Mục đích sử dụng gọi hàm 
- Cài đặt Cuộc gọi hàm bằng dịch vụ Azure Open AI 
- Thiết kế các cuộc gọi hàm hiệu quả cho trường hợp sử dụng ứng dụng của bạn 


## Hiểu về Gọi Hàm 

Trong bài học này, chúng ta muốn xây dựng một tính năng cho startup giáo dục của mình cho phép người dùng sử dụng chatbot để tìm các khóa học kỹ thuật. Chúng tôi sẽ đề xuất các khóa học phù hợp với trình độ kỹ năng, vai trò hiện tại và công nghệ họ quan tâm. 

Để hoàn thành điều này, chúng ta sẽ sử dụng sự kết hợp của: 
 - `Azure Open AI` để tạo trải nghiệm trò chuyện cho người dùng
 - `Microsoft Learn Catalog API` để giúp người dùng tìm khóa học dựa trên yêu cầu của họ 
 - `Function Calling` để lấy truy vấn của người dùng và gửi nó tới một hàm để thực hiện yêu cầu API. 

Để bắt đầu, hãy cùng xem tại sao chúng ta lại muốn sử dụng gọi hàm ngay từ đầu: 


### Tại Sao Gọi Hàm 

Nếu bạn đã hoàn thành bất kỳ bài học nào khác trong khóa học này, có lẽ bạn đã hiểu sức mạnh của việc sử dụng Mô hình Ngôn ngữ Lớn (LLMs). Hy vọng bạn cũng có thể thấy một số hạn chế của chúng. 

Gọi Hàm là một tính năng của Dịch vụ Azure Open AI nhằm khắc phục các hạn chế sau: 
1) Định dạng phản hồi nhất quán 
2) Khả năng sử dụng dữ liệu từ các nguồn khác của một ứng dụng trong bối cảnh trò chuyện 

Trước khi có gọi hàm, các phản hồi từ LLM thường không có cấu trúc và không nhất quán. Các nhà phát triển phải viết mã xác thực phức tạp để đảm bảo họ có thể xử lý từng biến thể của phản hồi. 

Người dùng không thể nhận được câu trả lời như "Thời tiết hiện tại ở Stockholm như thế nào?". Điều này là do các mô hình bị giới hạn ở thời điểm dữ liệu được huấn luyện. 

Hãy xem ví dụ bên dưới minh họa cho vấn đề này: 

Giả sử chúng ta muốn tạo một cơ sở dữ liệu dữ liệu sinh viên để có thể đề xuất khóa học phù hợp cho họ. Dưới đây là hai mô tả về sinh viên rất giống nhau trong dữ liệu mà chúng chứa.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Chúng tôi muốn gửi cái này đến một LLM để phân tích dữ liệu. Cái này sau đó có thể được sử dụng trong ứng dụng của chúng tôi để gửi đến một API hoặc lưu trữ trong cơ sở dữ liệu. 

Hãy tạo hai câu lệnh nhắc nhở giống hệt nhau mà chúng tôi chỉ dẫn LLM về những thông tin mà chúng tôi quan tâm: 


Chúng tôi muốn gửi điều này đến một LLM để phân tích các phần quan trọng đối với sản phẩm của chúng tôi. Vì vậy, chúng tôi có thể tạo hai lời nhắc giống hệt nhau để hướng dẫn LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Sau khi tạo hai lời nhắc này, chúng tôi sẽ gửi chúng đến LLM bằng cách sử dụng `client.responses.create`.  Chúng tôi lưu lời nhắc trong biến `input` và gán vai trò là `user`. Điều này để mô phỏng một tin nhắn từ người dùng được gửi đến chatbot. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Bây giờ chúng ta có thể gửi cả hai yêu cầu đến LLM và kiểm tra phản hồi mà chúng ta nhận được. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Mặc dù các prompt giống nhau và các mô tả tương tự, chúng ta có thể nhận được các định dạng khác nhau của thuộc tính `Grades`. 

Nếu bạn chạy ô trên nhiều lần, định dạng có thể là `3.7` hoặc `3.7 GPA`. 

Điều này là do LLM lấy dữ liệu không có cấu trúc dưới dạng prompt đã viết và cũng trả về dữ liệu không có cấu trúc. Chúng ta cần có một định dạng có cấu trúc để biết được thứ gì cần mong đợi khi lưu trữ hoặc sử dụng dữ liệu này.

Bằng cách sử dụng gọi hàm, chúng ta có thể đảm bảo nhận lại dữ liệu có cấu trúc. Khi sử dụng gọi hàm, LLM thực sự không gọi hoặc chạy bất kỳ hàm nào. Thay vào đó, chúng ta tạo một cấu trúc để LLM tuân theo trong phản hồi của nó. Sau đó, chúng ta sử dụng các phản hồi có cấu trúc đó để biết hàm nào cần chạy trong các ứng dụng của mình.  
 


![Sơ đồ luồng gọi hàm](../../../../translated_images/vi/Function-Flow.083875364af4f4bb.webp)


Chúng ta sau đó có thể lấy những gì được trả về từ hàm và gửi lại cho LLM. LLM sẽ phản hồi bằng ngôn ngữ tự nhiên để trả lời truy vấn của người dùng.


### Các trường hợp sử dụng cho việc gọi hàm 

**Gọi công cụ bên ngoài** 
Chatbot rất hữu ích trong việc cung cấp câu trả lời cho các câu hỏi từ người dùng. Bằng cách sử dụng gọi hàm, chatbot có thể sử dụng tin nhắn từ người dùng để hoàn thành các tác vụ nhất định. Ví dụ, một sinh viên có thể yêu cầu chatbot "Gửi email cho giảng viên của tôi nói rằng tôi cần thêm trợ giúp với môn học này". Điều này có thể gọi một hàm `send_email(to: string, body: string)`


**Tạo truy vấn API hoặc cơ sở dữ liệu**
Người dùng có thể tìm kiếm thông tin sử dụng ngôn ngữ tự nhiên được chuyển đổi thành truy vấn định dạng hoặc yêu cầu API. Một ví dụ có thể là một giáo viên yêu cầu "Ai là những sinh viên đã hoàn thành bài tập cuối cùng" có thể gọi một hàm tên là `get_completed(student_name: string, assignment: int, current_status: string)`


**Tạo dữ liệu có cấu trúc**
Người dùng có thể lấy một đoạn văn bản hoặc CSV và sử dụng LLM để trích xuất thông tin quan trọng từ đó. Ví dụ, một sinh viên có thể chuyển đổi một bài viết trên Wikipedia về các hiệp định hòa bình để tạo thẻ học AI. Điều này có thể được thực hiện bằng cách sử dụng một hàm gọi là `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Tạo Cuộc Gọi Hàm Đầu Tiên Của Bạn 

Quá trình tạo một cuộc gọi hàm bao gồm 3 bước chính: 
1. Gọi API Hoàn Thành Chat với danh sách các hàm của bạn và một tin nhắn từ người dùng 
2. Đọc phản hồi của mô hình để thực hiện một hành động ví dụ như thực thi một hàm hoặc gọi API 
3. Thực hiện một cuộc gọi khác đến API Hoàn Thành Chat với phản hồi từ hàm của bạn để sử dụng thông tin đó tạo phản hồi cho người dùng. 


![Luồng của một Cuộc gọi Hàm](../../../../translated_images/vi/LLM-Flow.3285ed8caf4796d7.webp)


### Các thành phần của một cuộc gọi hàm 

#### Đầu vào của người dùng 

Bước đầu tiên là tạo một tin nhắn của người dùng. Điều này có thể được gán động bằng cách lấy giá trị của một trường nhập văn bản hoặc bạn có thể gán giá trị ngay tại đây. Nếu đây là lần đầu bạn làm việc với Chat Completions API, chúng ta cần định nghĩa `role` và `content` của tin nhắn. 

`role` có thể là `system` (tạo quy tắc), `assistant` (mô hình) hoặc `user` (người dùng cuối). Đối với cuộc gọi hàm, chúng ta sẽ gán là `user` và một câu hỏi ví dụ. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Tạo hàm. 

Tiếp theo chúng ta sẽ định nghĩa một hàm và các tham số của hàm đó. Chúng ta sẽ sử dụng chỉ một hàm ở đây gọi là `search_courses` nhưng bạn có thể tạo nhiều hàm.

**Quan trọng** : Các hàm được bao gồm trong tin nhắn hệ thống gửi đến LLM và sẽ được tính vào số lượng token khả dụng bạn có.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Định nghĩa** 

`name` - Tên của hàm mà chúng ta muốn gọi. 

`description` - Đây là mô tả về cách hàm hoạt động. Ở đây quan trọng là phải cụ thể và rõ ràng 

`parameters` - Một danh sách các giá trị và định dạng mà bạn muốn mô hình tạo ra trong phản hồi của nó 


`type` - Kiểu dữ liệu mà các thuộc tính sẽ được lưu trữ trong đó 

`properties` - Danh sách các giá trị cụ thể mà mô hình sẽ sử dụng để phản hồi 


`name` - tên của thuộc tính mà mô hình sẽ sử dụng trong phản hồi đã định dạng của nó 

`type` - Kiểu dữ liệu của thuộc tính này 

`description` - Mô tả về thuộc tính cụ thể 


**Tùy chọn**

`required` - thuộc tính bắt buộc để cuộc gọi hàm được hoàn thành 


### Gọi hàm
Sau khi định nghĩa một hàm, bây giờ chúng ta cần đưa nó vào trong lời gọi đến API Hoàn thành Chat. Chúng ta làm điều này bằng cách thêm `functions` vào yêu cầu. Trong trường hợp này là `functions=functions`.

Cũng có một tùy chọn để đặt `function_call` thành `auto`. Điều này có nghĩa là chúng ta sẽ để LLM quyết định hàm nào nên được gọi dựa trên tin nhắn của người dùng thay vì chỉ định nó một cách thủ công.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Bây giờ chúng ta hãy xem phản hồi và xem nó được định dạng như thế nào: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Bạn có thể thấy rằng tên của hàm được gọi và từ tin nhắn của người dùng, LLM đã có thể tìm dữ liệu phù hợp với các đối số của hàm. 


## 3.Tích hợp cuộc gọi chức năng vào một ứng dụng. 


Sau khi chúng ta đã kiểm tra câu trả lời được định dạng từ LLM, bây giờ chúng ta có thể tích hợp điều này vào một ứng dụng. 

### Quản lý luồng 

Để tích hợp điều này vào ứng dụng của chúng ta, hãy thực hiện các bước sau: 

Đầu tiên, hãy thực hiện cuộc gọi đến dịch vụ Open AI và lưu tin nhắn vào một biến có tên là `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Bây giờ chúng ta sẽ định nghĩa hàm sẽ gọi API Microsoft Learn để lấy danh sách các khóa học: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Theo thực hành tốt nhất, chúng ta sẽ xem liệu mô hình có muốn gọi một hàm hay không. Sau đó, chúng ta sẽ tạo một trong các hàm có sẵn và khớp nó với hàm đang được gọi.
Tiếp theo, chúng ta sẽ lấy các đối số của hàm và ánh xạ chúng với các đối số từ LLM.

Cuối cùng, chúng ta sẽ thêm thông điệp gọi hàm cùng với các giá trị được trả về bởi thông điệp `search_courses`. Điều này cung cấp cho LLM tất cả thông tin cần thiết để
phản hồi người dùng bằng ngôn ngữ tự nhiên.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Bây giờ chúng ta sẽ gửi thông điệp đã cập nhật đến LLM để chúng ta có thể nhận phản hồi bằng ngôn ngữ tự nhiên thay vì phản hồi định dạng JSON của API. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Thử Thách Mã

Làm tốt lắm! Để tiếp tục học về Azure Open AI Function Calling, bạn có thể xây dựng: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst
 - Thêm nhiều tham số hơn cho hàm có thể giúp người học tìm thêm các khóa học. Bạn có thể tìm các tham số API có sẵn ở đây:
 - Tạo một cuộc gọi hàm khác nhận thêm thông tin từ người học như ngôn ngữ mẹ đẻ của họ
 - Tạo xử lý lỗi khi cuộc gọi hàm và/hoặc cuộc gọi API không trả về bất kỳ khóa học phù hợp nào


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
